# Dispersion curve from `pinn_data.mat`
Load PINN response and compute an f-k style dispersion map.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat

In [ ]:
mat = loadmat('../Result/pinn_data.mat')
print('keys:', [k for k in mat.keys() if not k.startswith('__')])

In [ ]:
# Adjust these field names if needed
# response shape expected: [n_time, n_space] or [n_space, n_time]
if 'x' in mat:
    X = np.array(mat['x'])
elif 'disp' in mat:
    X = np.array(mat['disp'])
else:
    raise KeyError('No displacement field found (expected x or disp).')

if X.shape[0] < X.shape[1]:
    X = X.T  # enforce [time, space]

X = X - X.mean(axis=0, keepdims=True)
nt, nx = X.shape
print('response shape [time, space]:', X.shape)

In [ ]:
# Optional defaults if dt/dx are not stored in MAT
dt = float(np.squeeze(mat['dt'])) if 'dt' in mat else 1.0
dx = float(np.squeeze(mat['dx'])) if 'dx' in mat else 1.0

S = np.fft.fftshift(np.fft.fft2(X))
P = np.abs(S)**2

f = np.fft.fftshift(np.fft.fftfreq(nt, d=dt))
k = 2*np.pi*np.fft.fftshift(np.fft.fftfreq(nx, d=dx))

In [ ]:
plt.figure(figsize=(8,5))
plt.pcolormesh(k, f, 10*np.log10(P / (P.max()+1e-12) + 1e-12), shading='auto', cmap='magma')
plt.xlabel('Wavenumber k [rad/m or index-based]')
plt.ylabel('Frequency f [Hz or index-based]')
plt.title('Dispersion-style f-k energy map from PINN response')
plt.colorbar(label='Power (dB, normalized)')
plt.tight_layout()
plt.show()